In [9]:
import pandas as pd
import numpy as np

def readData (dateiName):
    return pd.read_csv(dateiName)

In [4]:
def dfSauber(df):
    
    column_needed = ['regio1', 'streetPlain', 'houseNumber', 'regio3', 'geo_plz' ,'geo_krs', 'yearConstructed', 'baseRent',\
                     'noRooms', 'heatingCosts', 'serviceCharge', 'livingSpace', 'noParkSpaces',\
                     'balcony', 'hasKitchen', 'cellar', 'lift', 'garden', 'totalRent']
    df = df[column_needed]
    for col in df.columns:
        if df[col].any():
            df.dropna(subset=col, inplace=True)
    if df.duplicated().any():
        df.drop_duplicates(inplace=True)
    df["adress"] = df[["streetPlain", "houseNumber"]].apply(" ".join, axis=1)
    df['geo_plz']= df['geo_plz'].map(str)
    df["city"] = df[["geo_plz", "regio3"]].apply(" ".join, axis=1)
    df['full_address'] = df[["adress", "city", "regio1"]].apply(" ,".join, axis=1)
    df= df.drop(["streetPlain", "houseNumber", "geo_plz", "regio3", "regio1", "adress", "city"], axis='columns')

   

    return df
    

In [14]:
def service_geocode(address):
    location = geocode(address)
    if location is not None:
      return (location.latitude, location.longitude)
    else:
      return np.NaN

In [ ]:
#path ='/kaggle/input/immo-data-rental/immo_data.csv'
path = '../data/immo_data.csv'
df= readData(path)
#df.head()

## Bereinigung von datei
überprüfen welche column wir brauchen:

Adresse(streetPlain + housNumber) / Stadtteil (regio3) / PLZ(geo_plz)

bundesland (regio1), landkreis(regio_ksr)

serviceCharge(electricity or internet €)

Wohnfläche (m²)(livingSpace), Zimmeranzahl(noRooms), keller(cellar)

Baujahr(yearConstructed) garten (garden)

Ausstattung (Balkon(balkony), Aufzug(lift), Einbauküche(hasKitchen), Garage(noParkSpaces))

Kaltmiete (Preis) (baseRent)

Optional: Nebenkosten(heatingCosts/month), Warmmiete (totalRent (serviceCharge, heatingCosts,baseRent))


In [ ]:
datei = dfSauber(df)
#datei.head()

In [15]:
#address1 = datei['full_address'].iloc[0]
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
geolocator_arcgis = ArcGIS(timeout=20)
geocode = RateLimiter(geolocator_arcgis.geocode, min_delay_seconds=1, return_value_on_exception=None)
#location = geolocator_arcgis.geocode(address1)
#print('Latitude: '+str(location.latitude)+', Longitude: '+str(location.longitude))

In [16]:
datei['adresseInLatLon'] = datei['full_address'].apply(service_geocode)
datei[['full_address', 'adresseInLatLon']].head()

,full_address,adresseInLatLon
14,"Am_Dimberg 4 ,44229 Kirchhörde ,Nordrhein_West...","(51.457584985279, 7.455573960885)"
19,"Robert-Gernhardt-Platz 3 ,37073 Göttingen ,Nie...","(51.535960012831, 9.933195975578)"
29,"Zum_Bahnhof 7 ,19055 Paulsstadt ,Mecklenburg_V...","(53.633845017121, 11.409782968187)"
58,"Turnerstraße 27 ,4435 Schkeuditz ,Sachsen","(51.398860407713, 12.219154626233)"
68,"Kurfürstenstr. 25 ,65439 Flörsheim_am_Main ,He...","(50.019472981812, 8.424950031762)"


In [18]:
datei.to_csv("immo_data_with_lat_lon.csv", index=False)

In [ ]:

df.head()